# NCERT Multimodal RAG

Extracts text + images from 1096 NCERT chapter PDFs, indexes with Gemini embeddings, and lets you query with Gemini 3 Flash.

**Your only setup:** Paste your Gemini API key below. Get one free at https://aistudio.google.com/app/apikey

In [ ]:
GEMINI_API_KEY = ""  # @param {type:"string"}
MOUNT_DRIVE = True  # @param {type:"boolean"}

## 1. Install dependencies

In [ ]:
import subprocess, sys, os, json, re, time, pickle, hashlib
from pathlib import Path

reqs = [
    'google-genai',
    'faiss-cpu',
    'gradio',
    'PyMuPDF',
    'requests',
    'pillow',
    'tqdm',
    'numpy',
]
for r in reqs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', r], capture_output=True)

import numpy as np
import fitz
import gradio as gr
import requests
from PIL import Image
from tqdm.notebook import tqdm
from google import genai

client = genai.Client(api_key=GEMINI_API_KEY)
print('All imports OK')

## 2. Mount Google Drive (persistent cache)

In [ ]:
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/ncert_rag'
else:
    BASE = '/content/ncert_rag'
Path(BASE).mkdir(parents=True, exist_ok=True)
print(f'Working dir: {BASE}')

## 3. Load chapter index

The chapter index maps every PDF URL to its metadata (standard, subject, book, chapter title).

Auto-downloads from GitHub; falls back to manual upload.

In [ ]:
import json, requests
idx_path = Path(BASE) / 'chapter_index.json'

if not idx_path.exists():
    url = 'https://raw.githubusercontent.com/AshutoshGitMirror/ncert-rag/main/chapter_index.json'
    try:
        r = requests.get(url, timeout=15)
        if r.status_code == 200:
            with open(idx_path, 'wb') as f:
                f.write(r.content)
            print('Downloaded chapter_index.json from GitHub')
        else:
            raise Exception(f'HTTP {r.status_code}')
    except Exception as e:
        print(f'Download failed ({e}), prompting manual upload...')
        from google.colab import files
        uploaded = files.upload()
        for fn in uploaded:
            with open(idx_path, 'wb') as f:
                f.write(uploaded[fn])

with open(idx_path) as f:
    chapters = json.load(f)

print(f'Loaded {len(chapters)} chapters')
print(f'Sample: {json.dumps(chapters[0], indent=2)}')

## 4. Download PDFs (resumable)

Each chapter PDF is saved with a unique filename derived from the URL (e.g. `aemr101.pdf`).

In [ ]:
PDF_DIR = Path(BASE) / 'pdfs'
PDF_DIR.mkdir(exist_ok=True)

def url_to_uid(url):
    """Extract unique filename from PDF URL, e.g. 'aemr101' from '.../aemr101.pdf'"""
    return Path(url).stem

def download_pdfs(chapters, max_retries=3):
    existing = {f.stem for f in PDF_DIR.glob('*.pdf')}
    todo = [c for c in chapters if url_to_uid(c['url']) not in existing]
    print(f'Cached: {len(existing)}, To download: {len(todo)}')
    for c in tqdm(todo):
        uid = url_to_uid(c['url'])
        path = PDF_DIR / f'{uid}.pdf'
        for attempt in range(max_retries):
            try:
                r = requests.get(c['url'], timeout=30)
                if r.status_code == 200:
                    with open(path, 'wb') as f:
                        f.write(r.content)
                    break
            except:
                if attempt == max_retries - 1:
                    print(f'FAILED: {uid}')
                time.sleep(2)

download_pdfs(chapters)

## 5. Extract text + images from PDFs

Extracts text from every page and saves embedded images as PNGs.

In [ ]:
EXTRACT_DIR = Path(BASE) / 'extracted'
IMG_DIR = Path(BASE) / 'images'
EXTRACT_DIR.mkdir(exist_ok=True)
IMG_DIR.mkdir(exist_ok=True)

def extract_pdf(uid):
    """Extract text and images from one PDF. Returns page list. Caches to disk."""
    pdf_path = PDF_DIR / f'{uid}.pdf'
    if not pdf_path.exists():
        return None
    out_path = EXTRACT_DIR / f'{uid}.json'
    if out_path.exists():
        return json.load(open(out_path))
    doc = fitz.open(pdf_path)
    pages = []
    for page_num, page in enumerate(doc):
        text = page.get_text()
        images = []
        for img_idx, img in enumerate(page.get_images(full=True)):
            xref = img[0]
            bbox = page.get_image_bbox(img)
            if bbox is None or bbox.width < 50 or bbox.height < 50:
                continue
            pix = fitz.Pixmap(doc, xref)
            if pix.n > 4:
                pix = fitz.Pixmap(fitz.csRGB, pix)
            img_name = f'{uid}_p{page_num+1}_{img_idx}.png'
            pix.save(str(IMG_DIR / img_name))
            images.append(img_name)
        if text.strip() or images:
            pages.append({'page': page_num + 1, 'text': text, 'images': images})
    doc.close()
    result = {'code': uid, 'pages': pages}
    with open(out_path, 'w') as f:
        json.dump(result, f)
    return result

existing = {f.stem for f in EXTRACT_DIR.glob('*.json')}
todo = [c for c in chapters if url_to_uid(c['url']) not in existing]
print(f'Extracted: {len(existing)}, To process: {len(todo)}')
for c in tqdm(todo):
    extract_pdf(url_to_uid(c['url']))

## 6. Chunk text into sections

In [ ]:
def chunk_pages(pages, uid, std, subj, book, ch_title, ch_num, min_chars=500):
    chunks = []
    buffer = ''
    buffer_images = []
    buffer_pages = set()
    for p in pages:
        para = p['text'].strip()
        if not para:
            continue
        buffer += para + '\n\n'
        buffer_images.extend(p['images'])
        buffer_pages.add(p['page'])
        if len(buffer) >= min_chars:
            chunks.append({
                'text': buffer.strip(),
                'images': buffer_images,
                'pages': sorted(buffer_pages),
                'uid': uid, 'std': std, 'subj': subj,
                'book': book, 'ch': ch_title, 'ch_num': ch_num,
            })
            buffer = ''; buffer_images = []; buffer_pages = set()
    if buffer.strip():
        chunks.append({
            'text': buffer.strip(),
            'images': buffer_images,
            'pages': sorted(buffer_pages),
            'uid': uid, 'std': std, 'subj': subj,
            'book': book, 'ch': ch_title, 'ch_num': ch_num,
        })
    return chunks

CHUNKS_PATH = Path(BASE) / 'chunks.pkl'
if CHUNKS_PATH.exists():
    with open(CHUNKS_PATH, 'rb') as f:
        all_chunks = pickle.load(f)
    print(f'Loaded {len(all_chunks)} chunks from cache')
else:
    all_chunks = []
    for c in tqdm(chapters):
        uid = url_to_uid(c['url'])
        ext_path = EXTRACT_DIR / f'{uid}.json'
        if not ext_path.exists():
            continue
        with open(ext_path) as f:
            ext = json.load(f)
        chunks = chunk_pages(
            ext['pages'], uid, c['std'], c['subj'],
            c['book'], c['ch'], c['ch_num']
        )
        all_chunks.extend(chunks)
    with open(CHUNKS_PATH, 'wb') as f:
        pickle.dump(all_chunks, f)
    print(f'Created {len(all_chunks)} chunks')
print(f'Total chunks: {len(all_chunks)}')

## 7. Create embeddings -> FAISS index

Uses `gemini-embedding-2` (latest, 3072 dims, reduced to 768 for speed).

In [ ]:
INDEX_PATH = Path(BASE) / 'faiss.index'
META_PATH = Path(BASE) / 'chunks_meta.pkl'
EMBED_DIM = 768

def get_embedding(texts):
    result = client.models.embed_content(
        model='gemini-embedding-2',
        contents=texts,
        config={'output_dimensionality': EMBED_DIM},
    )
    return [e.values for e in result.embeddings]

if INDEX_PATH.exists():
    import faiss
    index = faiss.read_index(str(INDEX_PATH))
    with open(META_PATH, 'rb') as f:
        chunks_meta = pickle.load(f)
    print(f'Loaded index with {index.ntotal} vectors')
else:
    import faiss
    index = faiss.IndexFlatIP(EMBED_DIM)
    chunks_meta = []
    batch_size = 10
    for i in tqdm(range(0, len(all_chunks), batch_size)):
        batch = all_chunks[i:i+batch_size]
        texts = [c['text'][:2000] for c in batch]
        try:
            embs = get_embedding(texts)
            index.add(np.array(embs, dtype=np.float32))
            chunks_meta.extend(batch)
            time.sleep(0.5)
        except Exception as e:
            print(f'Batch {i} failed: {e}')
            time.sleep(10)
    faiss.write_index(index, str(INDEX_PATH))
    with open(META_PATH, 'wb') as f:
        pickle.dump(chunks_meta, f)
    print(f'Indexed {index.ntotal} vectors')

## 8. Query function

Embeds the question, searches FAISS for relevant chunks, returns context + images.

In [ ]:
def query(question, top_k=5):
    q_emb = get_embedding([question])[0]
    q_vec = np.array([q_emb], dtype=np.float32)
    scores, indices = index.search(q_vec, top_k)
    context_parts = []
    images = []
    for score, idx in zip(scores[0], indices[0]):
        m = chunks_meta[idx]
        context_parts.append(
            f"[Std {m['std']} | {m['subj']} | {m['book']} | "
            f"Ch {m['ch_num']}: {m['ch']} | Pages {m['pages']}]\n{m['text']}"
        )
        for img_name in m['images']:
            img_path = IMG_DIR / img_name
            if img_path.exists():
                images.append(Image.open(img_path))
    context = '\n---\n'.join(context_parts)
    prompt = f"""You are a friendly tutor for school children.
Answer based on these NCERT textbook excerpts. Be clear, engaging, and age-appropriate.
Refer to any diagrams provided.

Question: {question}

Textbook excerpts:
{context}
"""
    if images:
        response = client.models.generate_content(
            model='gemini-3-flash-preview',
            contents=[prompt] + images[:3],
        )
    else:
        response = client.models.generate_content(
            model='gemini-3-flash-preview',
            contents=prompt,
        )
    return response.text, images[:3], context

## 9. Test query

In [ ]:
test_q = "What are the different types of forces?"
answer, imgs, ctx = query(test_q)
print(f'Q: {test_q}')
print(f'A: {answer[:600]}...')
print(f'Diagrams: {len(imgs)}')

## 10. Gradio UI

Run this cell, click the public link, and share with students!

In [ ]:
with gr.Blocks(title='NCERT Tutor', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# NCERT Tutor')
    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(height=500, label='Chat')
            msg = gr.Textbox(
                label='Your question',
                placeholder='e.g. How does photosynthesis work?',
            )
        with gr.Column(scale=1):
            gallery = gr.Gallery(label='Diagrams', columns=1, height=400)
    def respond(question, chat_history):
        answer, imgs, _ = query(question)
        chat_history.append((question, answer))
        return '', chat_history, imgs
    msg.submit(respond, [msg, chatbot], [msg, chatbot, gallery])

demo.launch(share=True)